# IF-Only SVD Non-Zero Rank Truncation Merge

This notebook is a focused follow-up to `06_singular_task_interference_rl.ipynb` for IF-only recovery tests.

Experiment policy:

- Define IF task vector only:
  - `Delta_if = theta_if - theta_base`
- Do **not** use Math task vector at all (Math contribution is fully removed).
- For tensor dimension `>= 2`, apply truncated SVD independently per parameter and keep top non-zero singular values by ratio `{1%, 2%, 5%, 10%, 20%}`.
- For tensor dimension `< 2` (LayerNorm vectors/scalars), save two variants:
  - `unpruned`: apply full IF delta directly (no SVD)
  - `pruned`: apply zero update (keep base parameter unchanged)

Update formula per parameter:

- If `ndim >= 2`:
  - `theta = theta_base + TruncSVD(Delta_if)`
- If `ndim < 2` and mode is `unpruned`:
  - `theta = theta_base + Delta_if`
- If `ndim < 2` and mode is `pruned`:
  - `theta = theta_base`

One checkpoint is saved for every `(rank_ratio, vector_scalar_mode)` combination.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


# --------------------------------------------------------------------------------------
# Runtime configuration
# --------------------------------------------------------------------------------------


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for IF-only SVD checkpoint generation.

    Args:
        base_model_id: Hugging Face model id/path used as the base checkpoint.
        if_model_path: Local path to IF fine-tuned checkpoint.
        output_root: Root directory where merged checkpoints and summaries are saved.
        rank_keep_percentages: Non-zero singular-value retention ratios in percent.
        vector_scalar_modes: Mode set for `ndim < 2` parameters.
            - `unpruned`: apply raw IF delta without SVD.
            - `pruned`: keep base parameter unchanged (drop IF delta).
        model_dtype_name: Loading dtype for model weights.
        svd_device: Device used for SVD (`cuda:0` or `cpu`).
        nonzero_rtol: Relative tolerance for deciding whether singular values are non-zero.
        nonzero_atol: Absolute tolerance floor for non-zero singular values.
        overwrite_existing: Whether to overwrite already existing output checkpoint directories.
    """

    base_model_id: str = "Qwen/Qwen3-1.7B"
    if_model_path: Path = Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
    )
    output_root: Path = Path(
        "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery"
    )
    rank_keep_percentages: Tuple[float, ...] = (1, 2, 5, 10, 20)
    vector_scalar_modes: Tuple[str, ...] = ("unpruned", )
    model_dtype_name: str = "float32"
    svd_device: str = "cuda:0"
    nonzero_rtol: float = 1e-6
    nonzero_atol: float = 1e-8
    overwrite_existing: bool = False


RUNTIME = RuntimeConfig()
RUNTIME.output_root.mkdir(parents=True, exist_ok=True)
(RUNTIME.output_root / "metadata").mkdir(parents=True, exist_ok=True)


def resolve_torch_dtype(dtype_name: str) -> torch.dtype:
    """Resolve user-friendly dtype name into torch dtype object.

    Args:
        dtype_name: Dtype name string.

    Returns:
        Torch dtype corresponding to `dtype_name`.

    Raises:
        ValueError: If dtype name is unsupported.
    """

    normalized = dtype_name.strip().lower()
    mapping = {
        "float16": torch.float16,
        "fp16": torch.float16,
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    if normalized not in mapping:
        raise ValueError(f"Unsupported dtype name: {dtype_name}")
    return mapping[normalized]


def resolve_svd_device(device_name: str) -> torch.device:
    """Resolve SVD execution device with safe CUDA fallback.

    Args:
        device_name: Requested device string.

    Returns:
        Torch device object. Falls back to CPU when CUDA is unavailable.
    """

    requested = device_name.strip().lower()
    if requested.startswith("cuda") and torch.cuda.is_available():
        return torch.device(device_name)
    return torch.device("cpu")


MERGE_DTYPE = resolve_torch_dtype(RUNTIME.model_dtype_name)
SVD_DEVICE = resolve_svd_device(RUNTIME.svd_device)


if not RUNTIME.if_model_path.exists():
    raise FileNotFoundError(f"Required IF checkpoint path does not exist: {RUNTIME.if_model_path}")

for pct in RUNTIME.rank_keep_percentages:
    if pct <= 0 or pct > 100:
        raise ValueError(f"rank_keep_percentages must be in [1, 100], got: {pct}")

for mode in RUNTIME.vector_scalar_modes:
    if mode not in {"unpruned", "pruned"}:
        raise ValueError(f"vector_scalar_modes supports only ['unpruned', 'pruned'], got: {mode}")

print(f"Output root: {RUNTIME.output_root}")
print(f"Merge dtype: {MERGE_DTYPE}")
print(f"SVD device: {SVD_DEVICE}")
print(f"Rank retention percentages: {RUNTIME.rank_keep_percentages}")
print(f"Vector/scalar modes: {RUNTIME.vector_scalar_modes}")


Output root: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery
Merge dtype: torch.float16
SVD device: cuda:0
Rank retention percentages: (1, 2, 5, 10, 20)
Vector/scalar modes: ('unpruned',)


In [3]:
# --------------------------------------------------------------------------------------
# Model loading and validation helpers
# --------------------------------------------------------------------------------------


def load_causal_lm_cpu(model_name_or_path: str | Path, torch_dtype: torch.dtype) -> AutoModelForCausalLM:
    """Load a CausalLM checkpoint on CPU in eval mode.

    Args:
        model_name_or_path: Model id or local path.
        torch_dtype: Loading dtype.

    Returns:
        Loaded CausalLM model on CPU.
    """

    model = AutoModelForCausalLM.from_pretrained(
        str(model_name_or_path),
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=torch_dtype,
        trust_remote_code=True,
    )
    model.eval()
    return model


def model_named_parameters_dict(model: AutoModelForCausalLM) -> Dict[str, torch.nn.Parameter]:
    """Convert `named_parameters()` iterator into a dictionary.

    Args:
        model: Input model.

    Returns:
        Mapping from parameter name to parameter tensor wrapper.
    """

    return dict(model.named_parameters())


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
) -> None:
    """Validate key/shape compatibility across base and IF checkpoints.

    Args:
        base_model: Base checkpoint model.
        if_model: IF fine-tuned model.

    Returns:
        None. Raises ValueError when any incompatibility is detected.
    """

    base_params = model_named_parameters_dict(base_model)
    if_params = model_named_parameters_dict(if_model)

    if set(base_params.keys()) != set(if_params.keys()):
        raise ValueError("Parameter key mismatch across base/if models.")

    for name, base_param in base_params.items():
        if base_param.shape != if_params[name].shape:
            raise ValueError(f"Shape mismatch (base vs if) at parameter: {name}")


def percentage_to_output_tag(percent: float) -> str:
    """Format percentage integer into stable output-directory tag.

    Args:
        percent: Integer percentage in [1, 100].

    Returns:
        String tag such as `top_1pct_nonzero_singular`.
    """

    return f"top_{percent:g}pct_nonzero_singular"


def vector_scalar_mode_suffix(mode: str) -> str:
    """Map vector/scalar mode name into output-directory suffix.

    Args:
        mode: One of `unpruned` or `pruned`.

    Returns:
        Directory-safe suffix string.
    """

    if mode == "unpruned":
        return "vector_scalar_unpruned"
    if mode == "pruned":
        return "vector_scalar_pruned"
    raise ValueError(f"Unsupported vector/scalar mode: {mode}")


In [4]:
# --------------------------------------------------------------------------------------
# SVD truncation and IF-only merge helpers
# --------------------------------------------------------------------------------------


def truncated_svd_reconstruct_nonzero_fraction(
    tensor: torch.Tensor,
    keep_fraction: float,
    svd_device: torch.device,
    nonzero_rtol: float,
    nonzero_atol: float,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Reconstruct a tensor using top fraction of non-zero singular values.

    Important implementation detail:
    - The paper condition applies SVD to 2D layers. This helper supports `ndim >= 2`
      by flattening dimensions 1..N into one axis, i.e. shape `[d0, d1, ..., dk]`
      is treated as matrix `[d0, d1*...*dk]`.

    Args:
        tensor: Input update tensor (`Delta_if`) with `ndim >= 2`.
        keep_fraction: Fraction in (0, 1], applied to non-zero singular values.
        svd_device: Device used for SVD (`cpu` or `cuda`).
        nonzero_rtol: Relative threshold ratio for non-zero singular values.
        nonzero_atol: Absolute threshold floor for non-zero singular values.

    Returns:
        Tuple `(reconstructed_tensor, stats)` where `stats` includes non-zero rank,
        kept rank, matrix dimensions, and retained energy ratio.

    Raises:
        ValueError: If tensor has dimension < 2.
    """

    if tensor.ndim < 2:
        raise ValueError(f"SVD reconstruction expects ndim >= 2, got shape={tuple(tensor.shape)}")

    original_shape = tuple(tensor.shape)
    rows = int(original_shape[0])
    cols = int(tensor.numel() // max(rows, 1))

    # We always compute in float32 for numerical stability during decomposition.
    matrix_fp32 = tensor.detach().to(torch.float32).reshape(rows, cols)

    if rows == 0 or cols == 0:
        empty_stats = {
            "rows": float(rows),
            "cols": float(cols),
            "min_dim": float(min(rows, cols)),
            "nonzero_rank": 0.0,
            "kept_rank": 0.0,
            "energy_retained": 1.0,
        }
        return torch.zeros_like(tensor, dtype=torch.float32), empty_stats

    # Attempt SVD on requested device first. If CUDA fails unexpectedly,
    # we retry on CPU so the run can proceed instead of hard failing.
    matrix_device = matrix_fp32.to(svd_device)
    try:
        u, s, vh = torch.linalg.svd(matrix_device, full_matrices=False)
    except RuntimeError:
        u, s, vh = torch.linalg.svd(matrix_fp32.cpu(), full_matrices=False)

    s_cpu = s.detach().to(torch.float32).cpu()
    if s_cpu.numel() == 0:
        empty_stats = {
            "rows": float(rows),
            "cols": float(cols),
            "min_dim": float(min(rows, cols)),
            "nonzero_rank": 0.0,
            "kept_rank": 0.0,
            "energy_retained": 1.0,
        }
        return torch.zeros_like(tensor, dtype=torch.float32), empty_stats

    max_sv = float(torch.max(s_cpu).item())
    nonzero_threshold = max(nonzero_atol, max_sv * nonzero_rtol)
    nonzero_rank = int(torch.sum(s_cpu > nonzero_threshold).item())

    if nonzero_rank <= 0:
        zero_stats = {
            "rows": float(rows),
            "cols": float(cols),
            "min_dim": float(min(rows, cols)),
            "nonzero_rank": 0.0,
            "kept_rank": 0.0,
            "energy_retained": 1.0,
        }
        return torch.zeros_like(tensor, dtype=torch.float32), zero_stats

    kept_rank = int(max(1, math.ceil(float(nonzero_rank) * float(keep_fraction))))
    kept_rank = int(min(kept_rank, nonzero_rank))

    # Compact reconstruction using top singular triplets only.
    u_k = u[:, :kept_rank]
    s_k = s[:kept_rank]
    vh_k = vh[:kept_rank, :]
    reconstructed_matrix = (u_k * s_k.unsqueeze(0)) @ vh_k

    sq = s_cpu * s_cpu
    total_energy = float(torch.sum(sq[:nonzero_rank]).item())
    kept_energy = float(torch.sum(sq[:kept_rank]).item())
    energy_retained = 1.0 if total_energy <= 0.0 else float(kept_energy / total_energy)

    stats = {
        "rows": float(rows),
        "cols": float(cols),
        "min_dim": float(min(rows, cols)),
        "nonzero_rank": float(nonzero_rank),
        "kept_rank": float(kept_rank),
        "energy_retained": float(energy_retained),
    }

    reconstructed_tensor = reconstructed_matrix.detach().to(torch.float32).cpu().reshape(original_shape)
    return reconstructed_tensor, stats


def apply_if_only_svd_merge_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    keep_percent: float,
    vector_scalar_mode: str,
    svd_device: torch.device,
    nonzero_rtol: float,
    nonzero_atol: float,
) -> Dict[str, Any]:
    """Apply IF-only update with SVD truncation for matrix parameters.

    Formula implemented per parameter `p`:
        Delta_if_p = theta_if_p - theta_base_p

        if ndim(p) >= 2:
            theta_merged_p = theta_base_p + TruncatedSVD(Delta_if_p, keep_fraction)

        if ndim(p) < 2 and vector_scalar_mode == 'unpruned':
            theta_merged_p = theta_base_p + Delta_if_p

        if ndim(p) < 2 and vector_scalar_mode == 'pruned':
            theta_merged_p = theta_base_p

    Args:
        base_model: Base model to overwrite in-place with merged parameters.
        if_model: IF model providing IF task vector.
        keep_percent: Non-zero singular-value retention percentage.
        vector_scalar_mode: Policy for `ndim < 2` tensors (`unpruned` or `pruned`).
        svd_device: Device used for SVD.
        nonzero_rtol: Relative tolerance for non-zero singular values.
        nonzero_atol: Absolute tolerance for non-zero singular values.

    Returns:
        Summary dictionary with parameter counts and retained-rank statistics.
    """

    if vector_scalar_mode not in {"unpruned", "pruned"}:
        raise ValueError(f"Unsupported vector/scalar mode: {vector_scalar_mode}")

    keep_fraction = float(keep_percent) / 100.0

    base_params = model_named_parameters_dict(base_model)
    if_params = model_named_parameters_dict(if_model)

    summary: Dict[str, Any] = {
        "keep_percent": keep_percent,
        "keep_fraction": float(keep_fraction),
        "vector_scalar_mode": str(vector_scalar_mode),
        "total_parameters": int(len(base_params)),
        "updated_parameters": 0,
        "skipped_non_floating": 0,
        "svd_applied_parameters": 0,
        "vector_scalar_parameters": 0,
        "vector_scalar_updated": 0,
        "vector_scalar_pruned": 0,
        "if_nonzero_rank_total": 0.0,
        "if_kept_rank_total": 0.0,
        "if_energy_retained_mean": 0.0,
    }

    if_energy_list = []

    with torch.no_grad():
        for name, base_param in tqdm(base_params.items(), total=len(base_params), desc=f"IF SVD top {keep_percent}% | {vector_scalar_mode}"):
            if name not in if_params:
                continue

            if base_param.shape != if_params[name].shape:
                raise ValueError(f"Shape mismatch while merging parameter: {name}")

            if not torch.is_floating_point(base_param.data):
                summary["skipped_non_floating"] += 1
                continue

            # Convert to float32 on CPU so update arithmetic is deterministic
            # across source checkpoint dtypes and devices.
            base_fp32 = base_param.detach().to(torch.float32).cpu()
            if_fp32 = if_params[name].detach().to(torch.float32).cpu()

            delta_if = if_fp32 - base_fp32

            if base_fp32.ndim >= 2:
                delta_if_recon, if_stats = truncated_svd_reconstruct_nonzero_fraction(
                    tensor=delta_if,
                    keep_fraction=keep_fraction,
                    svd_device=svd_device,
                    nonzero_rtol=nonzero_rtol,
                    nonzero_atol=nonzero_atol,
                )

                merged_delta = delta_if_recon

                summary["svd_applied_parameters"] += 1
                summary["if_nonzero_rank_total"] += float(if_stats["nonzero_rank"])
                summary["if_kept_rank_total"] += float(if_stats["kept_rank"])
                if_energy_list.append(float(if_stats["energy_retained"]))
            else:
                summary["vector_scalar_parameters"] += 1

                if vector_scalar_mode == "unpruned":
                    # Unpruned mode keeps full IF update for vectors/scalars.
                    merged_delta = delta_if
                    summary["vector_scalar_updated"] += 1
                else:
                    # Pruned mode removes IF update for vectors/scalars.
                    merged_delta = torch.zeros_like(delta_if)
                    summary["vector_scalar_pruned"] += 1

            merged_tensor = base_fp32 + merged_delta
            base_param.data.copy_(merged_tensor.to(dtype=base_param.dtype, device=base_param.device))
            summary["updated_parameters"] += 1

    svd_count = max(int(summary["svd_applied_parameters"]), 1)
    summary["if_mean_kept_rank_ratio"] = float(summary["if_kept_rank_total"] / max(summary["if_nonzero_rank_total"], 1.0))
    summary["if_energy_retained_mean"] = float(sum(if_energy_list) / svd_count)

    return summary


def save_json(data: Mapping[str, Any], path: Path) -> None:
    """Save dictionary as UTF-8 JSON file with indentation.

    Args:
        data: JSON-serializable mapping.
        path: Output JSON path.

    Returns:
        None.
    """

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(dict(data), f, ensure_ascii=False, indent=2)


In [5]:
# --------------------------------------------------------------------------------------
# Main execution: build and save one checkpoint per (rank percentage, vector/scalar mode)
# --------------------------------------------------------------------------------------

analysis_start_time = time.time()

print("Loading tokenizer and IF model once...")
tokenizer = AutoTokenizer.from_pretrained(RUNTIME.base_model_id, trust_remote_code=True)
if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
    tokenizer.pad_token = tokenizer.eos_token

if_model = load_causal_lm_cpu(RUNTIME.if_model_path, torch_dtype=MERGE_DTYPE)

# Validate compatibility once using a temporary base model.
base_model_for_validation = load_causal_lm_cpu(RUNTIME.base_model_id, torch_dtype=MERGE_DTYPE)
validate_parameter_compatibility(
    base_model=base_model_for_validation,
    if_model=if_model,
)
del base_model_for_validation
gc.collect()

run_rows = []

for keep_percent in RUNTIME.rank_keep_percentages:
    for vector_scalar_mode in RUNTIME.vector_scalar_modes:
        output_tag = (
            f"{percentage_to_output_tag(keep_percent)}_"
            f"{vector_scalar_mode_suffix(vector_scalar_mode)}"
        )
        output_dir = RUNTIME.output_root / output_tag

        if output_dir.exists() and not RUNTIME.overwrite_existing:
            print(f"[Skip] Output already exists and overwrite is disabled: {output_dir}")
            run_rows.append(
                {
                    "keep_percent": keep_percent,
                    "vector_scalar_mode": str(vector_scalar_mode),
                    "status": "skipped_existing",
                    "output_dir": str(output_dir),
                }
            )
            continue

        print("=" * 120)
        print(f"Building checkpoint for keep_percent={keep_percent}% | vector_scalar_mode={vector_scalar_mode}")

        # Each run starts from a fresh base checkpoint so each variant is independent.
        base_model = load_causal_lm_cpu(RUNTIME.base_model_id, torch_dtype=MERGE_DTYPE)

        merge_summary = apply_if_only_svd_merge_inplace(
            base_model=base_model,
            if_model=if_model,
            keep_percent=keep_percent,
            vector_scalar_mode=vector_scalar_mode,
            svd_device=SVD_DEVICE,
            nonzero_rtol=RUNTIME.nonzero_rtol,
            nonzero_atol=RUNTIME.nonzero_atol,
        )

        output_dir.mkdir(parents=True, exist_ok=True)
        base_model.save_pretrained(output_dir, safe_serialization=True)
        tokenizer.save_pretrained(output_dir)

        metadata = {
            "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
            "method": "if_only_svd_nonzero_rank_truncation_with_vector_scalar_mode",
            "formula_matrix": "theta = theta_base + TruncSVD(Delta_if)",
            "formula_vector_scalar_unpruned": "theta = theta_base + Delta_if",
            "formula_vector_scalar_pruned": "theta = theta_base",
            "base_model_id": RUNTIME.base_model_id,
            "if_model_path": str(RUNTIME.if_model_path),
            "output_dir": str(output_dir),
            "keep_percent": keep_percent,
            "keep_fraction": float(keep_percent / 100.0),
            "vector_scalar_mode": str(vector_scalar_mode),
            "svd_device": str(SVD_DEVICE),
            "nonzero_rtol": float(RUNTIME.nonzero_rtol),
            "nonzero_atol": float(RUNTIME.nonzero_atol),
            "merge_summary": merge_summary,
        }
        save_json(metadata, output_dir / "merge_metadata.json")

        run_rows.append(
            {
                "keep_percent": keep_percent,
                "vector_scalar_mode": str(vector_scalar_mode),
                "status": "saved",
                "output_dir": str(output_dir),
                "updated_parameters": int(merge_summary["updated_parameters"]),
                "svd_applied_parameters": int(merge_summary["svd_applied_parameters"]),
                "vector_scalar_parameters": int(merge_summary["vector_scalar_parameters"]),
                "vector_scalar_updated": int(merge_summary["vector_scalar_updated"]),
                "vector_scalar_pruned": int(merge_summary["vector_scalar_pruned"]),
                "if_mean_kept_rank_ratio": float(merge_summary["if_mean_kept_rank_ratio"]),
                "if_energy_retained_mean": float(merge_summary["if_energy_retained_mean"]),
            }
        )

        print(
            f"Saved IF-only checkpoint -> {output_dir} "
            f"| svd_params={merge_summary['svd_applied_parameters']} "
            f"| vector_mode={vector_scalar_mode}"
        )

        # Release per-run base model before next variant to control host RAM.
        del base_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary_df = pd.DataFrame(run_rows)
display(summary_df)

run_summary = {
    "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "runtime_config": {
        **asdict(RUNTIME),
        "if_model_path": str(RUNTIME.if_model_path),
        "output_root": str(RUNTIME.output_root),
    },
    "svd_device": str(SVD_DEVICE),
    "elapsed_seconds": float(time.time() - analysis_start_time),
    "runs": run_rows,
}

summary_json_path = RUNTIME.output_root / "metadata" / "if_only_svd_nonzero_rank_summary.json"
summary_csv_path = RUNTIME.output_root / "metadata" / "if_only_svd_nonzero_rank_summary.csv"

save_json(run_summary, summary_json_path)
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved run summary JSON: {summary_json_path}")
print(f"Saved run summary CSV: {summary_csv_path}")

# Final cleanup to keep notebook sessions stable in long-running environments.
del if_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All done.")


Loading tokenizer and IF model once...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.25it/s]


Building checkpoint for keep_percent=1% | vector_scalar_mode=unpruned


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.92it/s]
IF SVD top 1% | unpruned: 100%|██████████| 310/310 [01:04<00:00,  4.83it/s]


Saved IF-only checkpoint -> /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery/top_1pct_nonzero_singular_vector_scalar_unpruned | svd_params=197 | vector_mode=unpruned
Building checkpoint for keep_percent=2% | vector_scalar_mode=unpruned


...

Saved IF-only checkpoint -> /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery/top_10pct_nonzero_singular_vector_scalar_unpruned | svd_params=197 | vector_mode=unpruned
Building checkpoint for keep_percent=20% | vector_scalar_mode=unpruned


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.81it/s]
IF SVD top 20% | unpruned: 100%|██████████| 310/310 [01:02<00:00,  4.97it/s]


Saved IF-only checkpoint -> /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery/top_20pct_nonzero_singular_vector_scalar_unpruned | svd_params=197 | vector_mode=unpruned


,keep_percent,vector_scalar_mode,status,output_dir,updated_parameters,svd_applied_parameters,vector_scalar_parameters,vector_scalar_updated,vector_scalar_pruned,if_mean_kept_rank_ratio,if_energy_retained_mean
0,1,unpruned,saved,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...,310,197,113,113,0,0.010335,0.212306
1,2,unpruned,saved,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...,310,197,113,113,0,0.020101,0.281621
2,5,unpruned,saved,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...,310,197,113,113,0,0.050375,0.407215
3,10,unpruned,saved,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...,310,197,113,113,0,0.100180,0.529945
4,20,unpruned,saved,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...,310,197,113,113,0,0.200199,0.680741


Saved run summary JSON: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery/metadata/if_only_svd_nonzero_rank_summary.json
Saved run summary CSV: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/svd_nonzero_rank_if_only_recovery/metadata/if_only_svd_nonzero_rank_summary.csv
All done.
